In [8]:
import pandas as pd

In [9]:
df = pd.read_csv("Scouting_Database_Final.csv")
df.head()

,player,team,league,age,nation,pos,playing time_min,standard_sot%,standard_g/sh,team success_ppm,...,performance_crdy_per90,performance_crdr_per90,performance_fls_per90,performance_fld_per90,performance_off_per90,performance_crs_per90,performance_int_per90,performance_tklw_per90,standard_sh_per90,standard_sot_per90
0,Aaron Cresswell,West Ham,ENG-Premier League,34,ENG,DF,3495.0,3.70,0.000,1.296667,...,0.180258,0.0,0.386266,0.360515,0.077253,4.660944,1.004292,0.566524,0.360515,0.025751
1,Aaron Hickey,Brentford,ENG-Premier League,24-092,SCO,"DF,MF",3383.0,12.30,0.000,1.430000,...,0.345847,0.0,1.037541,1.622820,0.026604,1.010937,0.798108,1.143955,0.638487,0.106414
2,Aaron Ramsey,Nice,FRA-Ligue 1,31,WAL,MF,2111.0,30.30,0.025,0.975000,...,0.170535,0.0,1.279015,0.937944,0.170535,2.600663,0.810043,1.833254,1.193747,0.341071
3,Aaron Wan-Bissaka,Aston Villa,ENG-Premier League,28-288,COD,"DF,MF",8469.0,16.92,0.026,1.232000,...,0.116897,0.0,0.818278,0.541977,0.063762,1.540914,1.710946,1.296493,0.308183,0.085016
4,Aaron Zehnter,Wolfsburg,Unknown,20,GER,"DF,MF",1457.0,22.75,0.090,2.050000,...,0.000000,0.0,0.123542,0.000000,0.000000,2.223747,1.050103,0.617708,0.679478,0.308854


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2908 entries, 0 to 2907
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   player                  2908 non-null   object 
 1   team                    2908 non-null   object 
 2   league                  2908 non-null   object 
 3   age                     2908 non-null   object 
 4   nation                  2908 non-null   object 
 5   pos                     2908 non-null   object 
 6   playing time_min        2908 non-null   float64
 7   standard_sot%           2908 non-null   float64
 8   standard_g/sh           2908 non-null   float64
 9   team success_ppm        2908 non-null   float64
 10  performance_gls_per90   2908 non-null   float64
 11  performance_ast_per90   2908 non-null   float64
 12  performance_g-pk_per90  2908 non-null   float64
 13  performance_pk_per90    2908 non-null   float64
 14  performance_crdy_per90  2908 non-null   

Before i start building the model, I have noticed something unusual. And what's unusual is the age column. For example kuna age inakaa ivi: 24-092 meaning 24 years and 92 days. We just want the number of years. So I am going to change that before building the model

In [13]:
import numpy as np

def convert_age(age_val):
  if pd.isna(age_val):
    return age_val

  age_str = str(age_val).strip()
  if '-' in age_str:
    parts = age_str.split('-')
    years = float(parts[0])
    days = float(parts[1])
    total_days = years * 365 + days
    return total_days / 365

  try:
    return float(age_str)
  except ValueError:
    return np.nan

df["age"] = df["age"].apply(convert_age)
df["age"] = np.floor(df["age"])
df["age"] = df["age"].astype(int)
df.head()

,player,team,league,age,nation,pos,playing time_min,standard_sot%,standard_g/sh,team success_ppm,...,performance_crdy_per90,performance_crdr_per90,performance_fls_per90,performance_fld_per90,performance_off_per90,performance_crs_per90,performance_int_per90,performance_tklw_per90,standard_sh_per90,standard_sot_per90
0,Aaron Cresswell,West Ham,ENG-Premier League,34,ENG,DF,3495.0,3.70,0.000,1.296667,...,0.180258,0.0,0.386266,0.360515,0.077253,4.660944,1.004292,0.566524,0.360515,0.025751
1,Aaron Hickey,Brentford,ENG-Premier League,24,SCO,"DF,MF",3383.0,12.30,0.000,1.430000,...,0.345847,0.0,1.037541,1.622820,0.026604,1.010937,0.798108,1.143955,0.638487,0.106414
2,Aaron Ramsey,Nice,FRA-Ligue 1,31,WAL,MF,2111.0,30.30,0.025,0.975000,...,0.170535,0.0,1.279015,0.937944,0.170535,2.600663,0.810043,1.833254,1.193747,0.341071
3,Aaron Wan-Bissaka,Aston Villa,ENG-Premier League,28,COD,"DF,MF",8469.0,16.92,0.026,1.232000,...,0.116897,0.0,0.818278,0.541977,0.063762,1.540914,1.710946,1.296493,0.308183,0.085016
4,Aaron Zehnter,Wolfsburg,Unknown,20,GER,"DF,MF",1457.0,22.75,0.090,2.050000,...,0.000000,0.0,0.123542,0.000000,0.000000,2.223747,1.050103,0.617708,0.679478,0.308854


`FEATURE SCALING `

In [15]:
#now i select the feature columns that I want to normalize
metadata_cols = ['player', 'team', 'league', 'pos', 'nation', 'age', 'playing time_min']
feature_cols = [col for col in df.columns if col not in metadata_cols]

X = df[feature_cols].copy()

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Another thing to note is that in this case I am not splitting the data.
If i hide 20% of the players that i have, my model will not be able to recomend these players when I search the database. This is like we are doing unsupervised learning, not supervised, because we have to remember we don't have a target column.
So I am going to need 100% of my data so that the algorithm can calculate the true relative distance between every available player

`BUILDING THE COSINE MODEL`

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(X_scaled)
similarity_matrix.shape

(2908, 2908)

so what happens here is that the cosine_similarity module here takes my dataset and calculates the angle between each and every single player and every other player simultaneously
So instead of checking one player at a time , it checks all the players simultaneously

so that (2908,2908) basically means 2908 rows representing the players and 2908 columns still representing those same players again. So it's like a table of players comparing each other.

`EVALUATING THE MODEL`

When it comes to evaluating models like this, things like accuracy won't matter because this isn't some kind of supervised learning. In this case, I'm gonna need some domain expertise to see if the model is returning good and valid players

In [20]:
def find_similar_players(player_name, top_n=10):
  player_index = df[df["player"].str.lower()==player_name.lower()].index[0]

  #this one will add a new column showing the scores
  df["similarity_score"] = similarity_matrix[player_index]

  #this one here skips the first row because the first row will be the player himself
  matches = df.nlargest(top_n+1, "similarity_score").iloc[1:].copy()

  #this one converts the score to percentage, then converts to a string
  matches["similarity_score"] = (matches["similarity_score"]*100).round(2).astype(str)+"%"

  return matches[metadata_cols + ["similarity_score"]]


Here i am putting some sort of safety net for any kind of misspelling and i will use a python module called difflib to do that.

In [30]:
import difflib #This module helps in finding the closest match to the name
import unicodedata

# this function will help in removing the accents in names and lowercases
def clean_str(text):
    return unicodedata.normalize('NFD', str(text)).encode('ascii', 'ignore').decode('utf-8').lower().strip()

# Player query (handles typos like 'mbape' or missing accents like 'mbappe')
player_to_search = "Kylian mbape"

# 1. Clean player names in our dataset for fair comparison
cleaned_db_names = df["player"].apply(clean_str).tolist()
search_clean = clean_str(player_to_search)

# 2. Look for fuzzy matches (typos & partial names)
matches = difflib.get_close_matches(search_clean, cleaned_db_names, n=1, cutoff=0.5)

if matches:
    # Get the exact original name from the DataFrame
    matched_idx = cleaned_db_names.index(matches[0])
    exact_name = df.iloc[matched_idx]["player"]

    print(f"🔍 Found closest match: '{exact_name}'\n")

    # Run our original search function with the correct exact name
    results = find_similar_players(exact_name, top_n=10)
    display(results)
else:
    print(f"Error: Player '{player_to_search}' not found in the dataset.")
    print("\nSome available players:")
    print(df["player"].head().to_string(index=False))

🔍 Found closest match: 'Kylian Mbappé'



,player,team,league,pos,nation,age,playing time_min,similarity_score
704,Dušan Vlahović,Juventus,ITA-Serie A,FW,SRB,25,6984.0,97.4%
415,Callum Wilson,Brentford,ENG-Premier League,FW,ENG,34,4505.0,96.83%
885,Folarin Balogun,Monaco,FRA-Ligue 1,FW,USA,24,7536.0,96.57%
1021,Harry Kane,Bayern Munich,Unknown,FW,ENG,33,11130.0,96.23%
2734,Victor Osimhen,Napoli,ITA-Serie A,FW,NGA,24,4548.0,95.76%
2794,Wissam Ben Yedder,Monaco,FRA-Ligue 1,FW,FRA,32,4430.0,95.17%
217,Ansu Fati,Monaco,FRA-Ligue 1,"MF,FW",ESP,22,3290.0,94.79%
1343,Joselu,Real Madrid,ESP-La Liga,FW,ESP,33,4677.0,94.68%
989,Gonçalo Ramos,Milan,ITA-Serie A,FW,POR,25,4073.0,94.6%
500,Christopher Nkunku,RB Leipzig,Unknown,FW,FRA,28,4727.0,94.59%


So according to my domain knowledge of football, the model is performing well, because i can see that it's calculating the similarities really well. I am happy with the results it's giving me.
There are no anomalies. If i key in a defender then i won't get some forwards in my results and viceversa

`EXPORTING THE MODEL`

In [32]:
import pickle

model_artifacts = {
    "df":df,
    "feature_cols":feature_cols,
    "similarity_matrix":similarity_matrix,
    "scaler":scaler
}

with open("model_artifacts_outfield.pkl", "wb") as f:
  pickle.dump(model_artifacts, f)


print("Sucesss")

Sucesss


NOW I AM ALSO GOING TO CREATE A SIMILAR MODEL FOR THE GOAL KEEPERS

In [34]:
df_gk = pd.read_csv("gk_processed.csv")
df_gk.head()

,player,team,league,pos,nation,age,playing time_min,performance_save%,performance_cs%,penalty kicks_save%,performance_ga_per90,performance_sota_per90,performance_saves_per90,performance_cs_per90,penalty kicks_pksv_per90
0,Aaron Ramsdale,Newcastle,ENG-Premier League,GK,ENG,27,7664,64.125,22.3,7.15,1.538361,4.474165,2.935804,0.234864,0.023486
1,Aarón Escandell,Oviedo,ESP-La Liga,GK,ESP,29,3393,48.550,13.9,14.30,1.564987,5.437666,3.872679,0.265252,0.053050
2,Adrian Šemper,Pisa,ITA-Serie A,GK,CRO,27,2250,66.100,16.0,0.00,1.680000,4.960000,3.280000,0.160000,0.000000
3,Adrián,Real Betis,ESP-La Liga,GK,ESP,38,1800,75.300,7.9,0.00,1.400000,4.200000,2.800000,0.150000,0.000000
4,Agustín Marchesín,Celta Vigo,ESP-La Liga,GK,ARG,34,1710,59.200,31.6,50.00,1.526316,3.736842,2.210526,0.315789,0.105263


In [41]:
df_gk["age"] = df_gk["age"].apply(convert_age)
df_gk["age"] = np.floor(df_gk["age"])
df_gk["age"] = df_gk["age"].astype(int)
df_gk.head()

,player,team,league,pos,nation,age,playing time_min,performance_save%,performance_cs%,penalty kicks_save%,performance_ga_per90,performance_sota_per90,performance_saves_per90,performance_cs_per90,penalty kicks_pksv_per90,similarity_score
0,Aaron Ramsdale,Newcastle,ENG-Premier League,GK,ENG,27,7664,64.125,22.3,7.15,1.538361,4.474165,2.935804,0.234864,0.023486,-0.160795
1,Aarón Escandell,Oviedo,ESP-La Liga,GK,ESP,29,3393,48.550,13.9,14.30,1.564987,5.437666,3.872679,0.265252,0.053050,-0.718729
2,Adrian Šemper,Pisa,ITA-Serie A,GK,CRO,27,2250,66.100,16.0,0.00,1.680000,4.960000,3.280000,0.160000,0.000000,-0.347774
3,Adrián,Real Betis,ESP-La Liga,GK,ESP,38,1800,75.300,7.9,0.00,1.400000,4.200000,2.800000,0.150000,0.000000,0.408521
4,Agustín Marchesín,Celta Vigo,ESP-La Liga,GK,ARG,34,1710,59.200,31.6,50.00,1.526316,3.736842,2.210526,0.315789,0.105263,-0.074790


In [42]:
#now i select the feature columns that I want to normalize
gk_metadata_cols = ['player', 'team', 'league', 'pos', 'nation', 'age', 'playing time_min']
gk_feature_cols = [col for col in df_gk.columns if col not in gk_metadata_cols]

X_gk = df_gk[gk_feature_cols].copy()

In [43]:
gk_scaler = StandardScaler()
X_scaled_gk = scaler.fit_transform(X_gk)

In [44]:
similarity_matrix_gk = cosine_similarity(X_scaled_gk)
similarity_matrix_gk.shape

(235, 235)

In [45]:
def find_similar_gk(gk_name, top_n=10):
  gk_index = df_gk[df_gk["player"].str.lower()==gk_name.lower()].index[0]

  #this one will add a new column showing the scores
  df_gk["similarity_score"] = similarity_matrix_gk[gk_index]

  #this one here skips the first row because the first row will be the goalie himself
  gk_matches = df_gk.nlargest(top_n+1, "similarity_score").iloc[1:].copy()

  #this one converts the score to percentage, then converts to a string
  gk_matches["similarity_score"] = (gk_matches["similarity_score"]*100).round(2).astype(str)+"%"

  return gk_matches[gk_metadata_cols + ["similarity_score"]]

In [46]:
import difflib #This module helps in finding the closest match to the name
import unicodedata

# this function will help in removing the accents in names and lowercases
def clean_str_gk(text):
    return unicodedata.normalize('NFD', str(text)).encode('ascii', 'ignore').decode('utf-8').lower().strip()

# Player query (handles typos like 'mbape' or missing accents like 'mbappe')
gk_to_search = "courtois"

# 1. Clean player names in our dataset for fair comparison
cleaned_db_names_gk = df_gk["player"].apply(clean_str).tolist()
search_clean_gk = clean_str_gk(gk_to_search)

# 2. Look for fuzzy matches (typos & partial names)
gk_matches = difflib.get_close_matches(search_clean_gk, cleaned_db_names_gk, n=1, cutoff=0.5)

if gk_matches:
    # Get the exact original name from the DataFrame
    matched_idx_gk = cleaned_db_names_gk.index(gk_matches[0])
    exact_name_gk = df_gk.iloc[matched_idx_gk]["player"]

    print(f"🔍 Found closest match: '{exact_name_gk}'\n")

    # Run our original search function with the correct exact name
    results_gk = find_similar_gk(exact_name_gk, top_n=10)
    display(results_gk)
else:
    print(f"Error: Player '{gk_to_search}' not found in the dataset.")
    print("\nSome available players:")
    print(df_gk["player"].head().to_string(index=False))

🔍 Found closest match: 'Thibaut Courtois'



,player,team,league,pos,nation,age,playing time_min,similarity_score
41,Claudio Bravo,Real Betis,ESP-La Liga,GK,CHI,40,1710,97.65%
39,Christos Mandas,Lazio,ITA-Serie A,GK,GRE,24,1801,93.7%
59,Edoardo Corvi,Parma,ITA-Serie A,GK,ITA,25,1800,93.02%
16,Alisson,Liverpool,ENG-Premier League,GK,BRA,33,10968,92.41%
5,Aitor Fernández,Osasuna,ESP-La Liga,GK,ESP,34,2667,91.56%
95,Jan Oblak,Atlético Madrid,ESP-La Liga,GK,SVN,33,12212,91.23%
169,Nicola Leali,Genoa,ITA-Serie A,GK,ITA,32,4633,90.35%
130,Lukáš Hrádecký,Leverkusen,Unknown,GK,FIN,34,10258,90.21%
89,Ivan Provedel,Lazio,ITA-Serie A,GK,ITA,31,11152,89.45%
119,Kepa Arrizabalaga,Real Madrid,ESP-La Liga,GK,ESP,28,6643,88.73%


In [47]:
gk_artifacts = {
    "df": df_gk,
    "feature_cols": gk_feature_cols,
    "similarity_matrix": similarity_matrix_gk,
    "scaler": gk_scaler
}

with open ("gk_model_artifacts.pkl", "wb") as f:
  pickle.dump(gk_artifacts, f)

print("Success")

Success
